# 트랙 B: 부하–웰니스 통계 PoV

## 목적

본 노트북은 **훈련 부하 구조 지표(ACWR, Monotony)가 다음날 Hooper Index를 얼마나 잘 예측하는지** 통계적으로 평가한다.  
분석은 단순 OLS 회귀부터 시작하여 선수 간 변동성을 설명하는 혼합효과모형(Mixed Effects Model)으로 확장하며,  
단일변수(ACWR only) → 다중변수(ACWR + Monotony) → EWMA 변형까지 총 **4개 모형**을 비교한다.

## 모형 설계

| 모형 | 유형 | 고정효과 | 랜덤효과 |
|------|------|----------|----------|
| M1 | OLS | ACWR(Rolling) | — |
| M2 | Mixed Effects | ACWR(Rolling) | (1\|athlete) |
| M3 | Mixed Effects | ACWR(Rolling) + Monotony | (1\|athlete) |
| M4 | Mixed Effects | ACWR(EWMA) + Monotony | (1\|athlete) |

## 분석 흐름

1. 합성 데모 데이터 생성 (12명 × 120일)  
2. 지표 산출: `acwr_rolling`, `acwr_ewma`, `monotony`, `strain`  
3. 4개 모형 적합 및 진단  
4. AIC / BIC / MAE / RMSE / Cohen’s f² 비교  
5. 결과 해석 및 한계 논의

In [ ]:
# --------------------------------------------------------------------------
# 환경 설정 및 모듈 임포트
# --------------------------------------------------------------------------
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.metrics.acwr import (
    acwr_rolling, acwr_ewma,
    atl_rolling, ctl_rolling,
    atl_ewma, ctl_ewma,
)
from src.metrics.monotony_strain import (
    monotony, strain, srpe, hooper_index,
)

np.random.seed(42)
print('\ud658\uacbd \uc124\uc815 \uc644\ub8cc')

## 합성 데이터 설명

실제 현장 데이터를 사용하기 전 **분석 파이프라인의 타당성을 검증**하기 위해 합성 데이터를 생성한다.

### 설계 파라미터

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| 선수 수 | 12명 | athlete_id: A01–A12 |
| 관측 기간 | 120일 | 충분한 워밍업 기간 확보 |
| daily_load 평균 | 400 AU | sRPE 기반, 주말(토·일) 0.5× |
| β₀ (절편) | 12.0 | 기저 Hooper Index |
| β_acwr | 2.5 | ACWR → Hooper 계수 |
| β_monotony | 1.5 | Monotony → Hooper 계수 |
| σ_athlete | 1.5 | 선수 간 랜덤 절편 SD |
| σ_noise | 1.8 | 잔차 노이즈 SD |

**해석**: 높은 ACWR · Monotony → 다음날 더 높은 Hooper Index (양의 관계).  
이는 급성 과부하 또는 단조로운 부하 패턴이 주관적 피로감을 증가시킨다는 스포츠과학적 가설과 일치한다.

In [ ]:
# --------------------------------------------------------------------------
# 합성 데이터 생성
# --------------------------------------------------------------------------

# 참값 파라미터
N_ATHLETES = 12
N_DAYS = 120
BETA_0 = 12.0
BETA_ACWR = 2.5
BETA_MONOTONY = 1.5
SIGMA_ATHLETE = 1.5
SIGMA_NOISE = 1.8

# 선수별 랜덤 절편
athlete_ids = [f'A{i+1:02d}' for i in range(N_ATHLETES)]
athlete_effects = {aid: np.random.normal(0, SIGMA_ATHLETE) for aid in athlete_ids}

records = []

for aid in athlete_ids:
    # 일별 부하 생성: 평균 400 AU, 주말 0.5배
    base_load = np.random.normal(400, 80, size=N_DAYS).clip(50)
    days = np.arange(N_DAYS)
    # 요일: 0=월, 5=토, 6=일
    day_of_week = days % 7
    weekend_mask = (day_of_week >= 5)
    base_load[weekend_mask] *= 0.5

    loads_series = pd.Series(base_load, name='daily_load')

    # 지표 산출
    acwr_r = acwr_rolling(loads_series)       # Rolling ACWR
    acwr_e = acwr_ewma(loads_series)          # EWMA ACWR
    mono   = monotony(loads_series)           # Monotony
    strn   = strain(loads_series)             # Strain

    for d in range(N_DAYS):
        acwr_r_val = acwr_r.iloc[d]
        acwr_e_val = acwr_e.iloc[d]
        mono_val   = mono.iloc[d]
        strn_val   = strn.iloc[d]

        # hooper_next: 참값 모형으로 생성 (NaN 지표는 건너뛰)
        if np.isnan(acwr_r_val) or np.isnan(mono_val):
            hooper_next = np.nan
        else:
            hooper_next = (
                BETA_0
                + BETA_ACWR * acwr_r_val
                + BETA_MONOTONY * mono_val
                + athlete_effects[aid]
                + np.random.normal(0, SIGMA_NOISE)
            )

        records.append({
            'athlete_id': aid,
            'day': d,
            'daily_load': base_load[d],
            'acwr_rolling': acwr_r_val,
            'acwr_ewma': acwr_e_val,
            'monotony': mono_val,
            'strain': strn_val,
            'hooper_next': hooper_next,
        })

df = pd.DataFrame(records)

# NaN 제거 (워밍업 기간)
df_clean = df.dropna(subset=['acwr_rolling', 'acwr_ewma', 'monotony', 'hooper_next']).copy()
df_clean.reset_index(drop=True, inplace=True)

print(f'전체 레코드: {len(df):,}  |  분석 대상 (워밍업 제거 후): {len(df_clean):,}')
print(f'\uc120\uc218 \uc218: {df_clean.athlete_id.nunique()}')
print()
df_clean.describe().round(2)

## 모형 1 — OLS 베이스라인 (ACWR Rolling 단일변수)

가장 단순한 모형으로, 선수 간 변동성을 무시하고 Rolling ACWR만으로 다음날 Hooper Index를 예측한다.  
이 모형은 **비교 기준선(baseline)** 역할을 한다.

$$\text{Hooper}_{t+1} = \beta_0 + \beta_1 \cdot \text{ACWR}_{\text{rolling},t} + \epsilon_t$$

In [ ]:
# --------------------------------------------------------------------------
# 모형 1: OLS — hooper_next ~ acwr_rolling
# --------------------------------------------------------------------------
m1_ols = smf.ols('hooper_next ~ acwr_rolling', data=df_clean).fit()

# 진단 지표 추출
m1_pred = m1_ols.predict(df_clean)
m1_mae = mean_absolute_error(df_clean['hooper_next'], m1_pred)
m1_rmse = np.sqrt(mean_squared_error(df_clean['hooper_next'], m1_pred))

print('=== 모형 1: OLS (ACWR Rolling) ===')
print(m1_ols.summary())
print(f'\nR\u00b2 = {m1_ols.rsquared:.4f}')
print(f'AIC = {m1_ols.aic:.2f}')
print(f'BIC = {m1_ols.bic:.2f}')
print(f'MAE = {m1_mae:.4f}')
print(f'RMSE = {m1_rmse:.4f}')

## 모형 2 — 혼합효과모형: Rolling ACWR + (1|athlete)

선수별 랜덤 절편을 추가하여 선수 간 기저 피로도 차이를 설명한다.  
OLS 대비 잔차 분산이 감소하는지, AIC/BIC가 개선되는지 확인한다.

$$\text{Hooper}_{t+1} = (\beta_0 + u_j) + \beta_1 \cdot \text{ACWR}_{\text{rolling},t} + \epsilon_t, \quad u_j \sim N(0, \sigma_u^2)$$

In [ ]:
# --------------------------------------------------------------------------
# 모형 2: Mixed Effects — hooper_next ~ acwr_rolling, (1|athlete)
# --------------------------------------------------------------------------
m2_me = smf.mixedlm(
    'hooper_next ~ acwr_rolling',
    data=df_clean,
    groups='athlete_id',
).fit(reml=False)   # ML 추정 (AIC/BIC 비교용)

m2_pred = m2_me.predict(df_clean)
m2_mae = mean_absolute_error(df_clean['hooper_next'], m2_pred)
m2_rmse = np.sqrt(mean_squared_error(df_clean['hooper_next'], m2_pred))

print('=== 모형 2: Mixed Effects (ACWR Rolling) ===')
print(m2_me.summary())
print(f'\nAIC = {m2_me.aic:.2f}')
print(f'BIC = {m2_me.bic:.2f}')
print(f'MAE = {m2_mae:.4f}')
print(f'RMSE = {m2_rmse:.4f}')

# 잔차 진단 플롯
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

resid = df_clean['hooper_next'] - m2_pred

# 잔차 히스토그램
axes[0].hist(resid, bins=40, edgecolor='black', alpha=0.7)
axes[0].set_title('M2 Residual Distribution')
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Frequency')

# 잔차 vs 예측값
axes[1].scatter(m2_pred, resid, alpha=0.25, s=8)
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.8)
axes[1].set_title('M2 Residual vs Fitted')
axes[1].set_xlabel('Fitted')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 모형 3 — 혼합효과모형: ACWR Rolling + Monotony + (1|athlete)

**Monotony**를 추가 예측변수로 투입한다.  
훈련 단조성이 높으면(같은 강도로 반복) 회복 부담이 증가하여 Hooper가 올라간다는 가설을 검증한다.

$$\text{Hooper}_{t+1} = (\beta_0 + u_j) + \beta_1 \cdot \text{ACWR}_{\text{rolling},t} + \beta_2 \cdot \text{Monotony}_t + \epsilon_t$$

In [ ]:
# --------------------------------------------------------------------------
# 모형 3: Mixed Effects — hooper_next ~ acwr_rolling + monotony, (1|athlete)
# --------------------------------------------------------------------------
m3_me = smf.mixedlm(
    'hooper_next ~ acwr_rolling + monotony',
    data=df_clean,
    groups='athlete_id',
).fit(reml=False)

m3_pred = m3_me.predict(df_clean)
m3_mae = mean_absolute_error(df_clean['hooper_next'], m3_pred)
m3_rmse = np.sqrt(mean_squared_error(df_clean['hooper_next'], m3_pred))

print('=== 모형 3: Mixed Effects (ACWR Rolling + Monotony) ===')
print(m3_me.summary())
print(f'\nAIC = {m3_me.aic:.2f}')
print(f'BIC = {m3_me.bic:.2f}')
print(f'MAE = {m3_mae:.4f}')
print(f'RMSE = {m3_rmse:.4f}')

## 모형 4 — 혼합효과모형: EWMA ACWR + Monotony + (1|athlete)

Rolling ACWR 대신 **EWMA ACWR**을 사용한다.  
EWMA는 최근 부하에 더 높은 가중치를 부여하므로, 급성 부하 변화에 더 민감하게 반응할 수 있다.

$$\text{Hooper}_{t+1} = (\beta_0 + u_j) + \beta_1 \cdot \text{ACWR}_{\text{ewma},t} + \beta_2 \cdot \text{Monotony}_t + \epsilon_t$$

In [ ]:
# --------------------------------------------------------------------------
# 모형 4: Mixed Effects — hooper_next ~ acwr_ewma + monotony, (1|athlete)
# --------------------------------------------------------------------------
m4_me = smf.mixedlm(
    'hooper_next ~ acwr_ewma + monotony',
    data=df_clean,
    groups='athlete_id',
).fit(reml=False)

m4_pred = m4_me.predict(df_clean)
m4_mae = mean_absolute_error(df_clean['hooper_next'], m4_pred)
m4_rmse = np.sqrt(mean_squared_error(df_clean['hooper_next'], m4_pred))

print('=== 모형 4: Mixed Effects (EWMA ACWR + Monotony) ===')
print(m4_me.summary())
print(f'\nAIC = {m4_me.aic:.2f}')
print(f'BIC = {m4_me.bic:.2f}')
print(f'MAE = {m4_mae:.4f}')
print(f'RMSE = {m4_rmse:.4f}')

## 모형 비교

4개 모형의 적합도 지표와 효과크기를 종합적으로 비교한다.

### 비교 기준
- **AIC / BIC**: 낮을수록 우수 (절대적 해석 불가, 상대 비교용)
- **MAE / RMSE**: 예측 정확도. 낮을수록 우수
- **Cohen's f²**: 효과크기. 0.02(소), 0.15(중), 0.35(대)
- **계수 p-value**: 통계적 유의성 (단, p-value 단독으로 결론을 내리지 않음)

In [ ]:
# --------------------------------------------------------------------------
# 모형 비교 요약표 및 시각화
# --------------------------------------------------------------------------
import os

# --- Cohen's f² 계산 ---
# f² = (R²_full - R²_reduced) / (1 - R²_full)
# OLS R²를 기준으로, 각 모형의 pseudo-R² (1 - SS_res/SS_tot) 사용

y = df_clean['hooper_next'].values
ss_tot = np.sum((y - y.mean())**2)

def pseudo_r2(pred):
    ss_res = np.sum((y - pred.values)**2)
    return 1 - ss_res / ss_tot

r2_m1 = pseudo_r2(m1_pred)
r2_m2 = pseudo_r2(m2_pred)
r2_m3 = pseudo_r2(m3_pred)
r2_m4 = pseudo_r2(m4_pred)

# M1(OLS) 대비 각 모형의 Cohen's f²
def cohens_f2(r2_full, r2_reduced):
    denom = 1 - r2_full
    if denom <= 0:
        return np.nan
    return (r2_full - r2_reduced) / denom

# --- ACWR 계수/p-value 추출 ---
def extract_coef(model, param):
    """OLS 또는 MixedLM에서 계수와 p-value 추출"""
    try:
        coef = model.params[param]
        pval = model.pvalues[param]
    except (KeyError, AttributeError):
        coef, pval = np.nan, np.nan
    return coef, pval

acwr_coef_m1, acwr_p_m1 = extract_coef(m1_ols, 'acwr_rolling')
acwr_coef_m2, acwr_p_m2 = extract_coef(m2_me, 'acwr_rolling')
acwr_coef_m3, acwr_p_m3 = extract_coef(m3_me, 'acwr_rolling')
acwr_coef_m4, acwr_p_m4 = extract_coef(m4_me, 'acwr_ewma')

mono_coef_m3, mono_p_m3 = extract_coef(m3_me, 'monotony')
mono_coef_m4, mono_p_m4 = extract_coef(m4_me, 'monotony')

# --- 비교표 구성 ---
comparison = pd.DataFrame({
    'Model': ['M1: OLS (ACWR)', 'M2: Mixed (ACWR)',
              'M3: Mixed (ACWR+Mono)', 'M4: Mixed (EWMA+Mono)'],
    'AIC':  [m1_ols.aic, m2_me.aic, m3_me.aic, m4_me.aic],
    'BIC':  [m1_ols.bic, m2_me.bic, m3_me.bic, m4_me.bic],
    'MAE':  [m1_mae, m2_mae, m3_mae, m4_mae],
    'RMSE': [m1_rmse, m2_rmse, m3_rmse, m4_rmse],
    'R2':   [r2_m1, r2_m2, r2_m3, r2_m4],
    'ACWR_coef': [acwr_coef_m1, acwr_coef_m2, acwr_coef_m3, acwr_coef_m4],
    'ACWR_p':    [acwr_p_m1, acwr_p_m2, acwr_p_m3, acwr_p_m4],
    'Mono_coef': [np.nan, np.nan, mono_coef_m3, mono_coef_m4],
    'Mono_p':    [np.nan, np.nan, mono_p_m3, mono_p_m4],
    'f2_vs_M1':  [0.0, cohens_f2(r2_m2, r2_m1),
                  cohens_f2(r2_m3, r2_m1), cohens_f2(r2_m4, r2_m1)],
})

print('=== 4개 모형 비교 요약 ===')
print(comparison.to_string(index=False, float_format='%.4f'))

# --- AIC / BIC 막대 차트 ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
labels = comparison['Model'].tolist()
x = np.arange(len(labels))

# AIC
axes[0].bar(x, comparison['AIC'], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
axes[0].set_ylabel('AIC')
axes[0].set_title('AIC Comparison (lower is better)')
for i, v in enumerate(comparison['AIC']):
    axes[0].text(i, v + 10, f'{v:.0f}', ha='center', va='bottom', fontsize=8)

# BIC
axes[1].bar(x, comparison['BIC'], color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
axes[1].set_ylabel('BIC')
axes[1].set_title('BIC Comparison (lower is better)')
for i, v in enumerate(comparison['BIC']):
    axes[1].text(i, v + 10, f'{v:.0f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()

# 그림 저장
fig_dir = os.path.join('..', 'reports', 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'track_B_model_comparison.png')
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'\n그림 저장: {fig_path}')
plt.show()

## 해석 및 한계

### 주요 발견

1. **랜덤 절편의 효과**: OLS(M1) 대비 혼합효과모형(M2)은 선수 간 기저 피로 차이를 포착하여  
   AIC/BIC가 개선될 것으로 예상된다. 이는 선수별 개인차를 무시하면 추정 편향이 발생할 수 있음을 시사한다.

2. **Monotony 추가의 가치**: M2 → M3에서 Monotony를 추가하면 AIC/BIC가 추가 하락하고  
   MAE/RMSE가 감소할 것으로 예상된다. 이는 부하의 “양”(ACWR)뿐 아니라 “패턴”(Monotony)도  
   웰니스 변화에 기여함을 나타낸다.

3. **Rolling vs EWMA**: M3 vs M4 비교에서 EWMA ACWR은 최근 부하에 더 높은 가중치를 부여하므로  
   급성 과부하에 민감하게 반응할 수 있다. 그러나 본 합성 데이터는 Rolling ACWR로 생성되었으므로  
   M3가 M4보다 약간 우수할 가능성이 높다. 실제 데이터에서는 EWMA가 더 적합할 수 있다.

4. **효과크기**: Cohen’s f² 값은 모형 간 설명력 차이의 실질적 크기를 평가하는 데 사용된다.  
   p-value만으로는 표본 크기에 따라 해석이 달라질 수 있으므로, 효과크기를 병행 보고한다.

### 한계

- **합성 데이터**: 본 분석은 알려진 파라미터로 생성된 합성 데이터를 사용하였다.  
  실제 데이터에서는 비선형 관계, 교란변수, 결측 패턴 등이 결과에 영향을 줄 수 있다.
- **시간적 자기상관**: 현재 모형은 잔차의 시간적 자기상관(autocorrelation)을 고려하지 않는다.  
  AR(1) 구조 추가 또는 시계열 모형으로의 확장을 향후 검토할 필요가 있다.
- **랜덤 기울기**: 현재는 랜덤 절편만 포함하였으나, 선수별로 ACWR → Hooper 민감도가 다를 수 있다.  
  랜덤 기울기 모형(random slopes)을 후속 연구에서 검토할 수 있다.
- **교란변수**: 수면, 스트레스, 영양 상태 등 모형에 포함되지 않은 교란변수가 존재할 수 있다.
- **외적 타당도**: 12명 × 120일 합성 데이터의 결과를 실제 팀 환경에 직접 일반화하기 어렵다.

### 권장 사항

- 실제 데이터 수집 후 동일 파이프라인을 재실행하여 결과를 교차 검증할 것
- Monotony 지표의 추가적 가치가 실제 데이터에서도 확인되는지 재현 분석 수행
- 모형 선택 시 AIC/BIC뿐 아니라 교차검증(CV) 기반 예측 성능도 함께 보고할 것